# 03 — Override Pathway and Illustrative Application

**EPISTEMIC STATUS: EXPLANATORY** (with one clearly demarcated **ILLUSTRATIVE** section for the Epic Sepsis case)

This notebook walks through the constrained override pathway logic, override accumulation safeguards, and provides an illustrative application of the gate architecture to the Epic Sepsis Model case.

**Claim IDs (primary coverage in this notebook):** P1-C05 (override + shared service context), P1-C14, P1-C16, P1-C17.

**STM targets:** STM-CF3 (Override pathway), STM-CF6 (Epic Sepsis illustration)

**Outputs:** `outputs/tables/override_summary.csv`


## Override Pathway: Why It Exists

The conjunctive rule's principal advantage — that critical safety and equity gaps cannot be offset by aggregate performance — is also the source of its principal limitation: the risk of blocking beneficial tools with correctable evidence deficits.

The override pathway is not a weakening of the conjunctive rule but a structured mechanism for distinguishing safety-critical failures from governance-administrative deficits resolvable within a bounded period.

**Gates 1 (Safety) and 5 (Monitoring) remain non-overridable.** Only Gates 2 (Equity), 3 (Documentation), and 4 (Accountability) are eligible for override.


In [1]:
import json
import hashlib
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

# Load override specification
with open(REPO_ROOT / 'data' / 'override_specification.json', 'r', encoding='utf-8') as f:
    override_data = json.load(f)

# Render operative criteria
print("=== Four Operative Criteria (ALL must be simultaneously satisfied) ===")
for c in override_data['operative_criteria']:
    prov = f"  [{c['provisional_note']}]" if 'provisional_note' in c else ""
    print(f"  ({c['number']}) {c['criterion']}{prov}")


=== Four Operative Criteria (ALL must be simultaneously satisfied) ===
  (1) Documented rationale from the accountable clinical lead
  (2) Maximum validity period of 90 days  [Proposed as a provisional pilot default requiring empirical calibration rather than a quasi-canonical value]
  (3) Prospective monitoring plan specific to the gap
  (4) Board or governance committee approval producing a documented record


## Override Accumulation Safeguards

Three structural safeguards prevent override creep:


In [2]:
# Render safeguards and save summary
rows = []
for s in override_data['accumulation_safeguards']:
    prov = s.get('provisional_note', '')
    rows.append({
        'Safeguard': s['name'],
        'Description': s['description'],
        'Provisional Note': prov,
    })

for c in override_data['operative_criteria']:
    rows.append({
        'Safeguard': f"Criterion {c['number']}",
        'Description': c['criterion'],
        'Provisional Note': c.get('provisional_note', ''),
    })

df_override = pd.DataFrame(rows)
output_path = REPO_ROOT / 'outputs' / 'tables' / 'override_summary.csv'
df_override.to_csv(output_path, index=False)
print(f"Written: {output_path.relative_to(REPO_ROOT)}")

# Display safeguards
pd.DataFrame([{'Safeguard': s['name'], 'Description': s['description']} for s in override_data['accumulation_safeguards']])


Written: outputs\tables\override_summary.csv


,Safeguard,Description
0,Concurrent override limits,A single AI system may not operate under more ...
1,Institutional override registers,"All active overrides maintained in a single, a..."
2,Escalation triggers,If more than two AI systems are operating unde...


## Decision-Tree Walkthrough

```
Gate fails
  │
  ├── Is it Gate 1 or Gate 5?
  │     │
  │     └── YES → NON-OVERRIDABLE. Hard stop. Deployment blocked
  │               until evidence gap is fully remediated.
  │
  └── NO (Gate 2, 3, or 4)
        │
        ├── Are ALL 4 override criteria simultaneously satisfied?
        │     │
        │     ├── YES → Override GRANTED
        │     │         • Maximum 90 days validity (provisional default)
        │     │         • Prospective monitoring of the gap required
        │     │         • Documented in institutional override register
        │     │
        │     └── NO → Deployment BLOCKED until gap is remediated
        │
        └── Are concurrent override limits exceeded?
              │
              ├── >1 active override on same system → BLOCKED
              │
              └── >2 systems under override OR >30% of systems
                  trigger override on same gate → Governance
                  committee STRUCTURAL REVIEW required
```

### Provisional Defaults Note

> The 90-day override validity period, the concurrent override limit, and the 30% escalation trigger are proposed as provisional pilot defaults on practical grounds rather than empirical evidence; their optimal values are an explicit priority for empirical testing.
>
> — Paper 1 Manuscript


---

## ⚠️ ILLUSTRATIVE APPLICATION BOUNDARY

The following section maps a documented real-world case to the gate architecture for explanatory purposes. The manuscript explicitly states:

> "This case does not demonstrate that gate logic would have produced a categorically better patient outcome — that claim is not empirically testable retrospectively."

---


In [3]:
# Load Epic Sepsis illustration
with open(REPO_ROOT / 'data' / 'epic_sepsis_illustration.json', 'r', encoding='utf-8') as f:
    sepsis_data = json.load(f)

print(f"Case: {sepsis_data['case']}")
print(f"\nDocumented findings:")
for finding in sepsis_data['documented_findings']:
    print(f"  • {finding}")

print(f"\nGate mapping:")
for gm in sepsis_data['gate_mapping']:
    status_marker = "✗ FAIL" if gm['status'] == 'FAIL' else ("— NOT ASSESSED" if gm['status'] == 'NOT ASSESSED' else "✓ PASS")
    print(f"  Gate {gm['gate']} ({gm['domain']}): {status_marker}")
    print(f"    Reason: {gm['reason']}")

print(f"\nSummary: {sepsis_data['summary']}")


Case: Epic Sepsis Model

Documented findings:
  • External validation identified materially lower performance than vendor claims [21]
  • Clinician alert burden was identified as a significant operational concern [22]

Gate mapping:
  Gate 1 (Clinical Safety and Validation): ✗ FAIL
    Reason: Lack of local calibration; materially lower performance than vendor claims in external validation
  Gate 2 (Bias and Equity Oversight): — NOT ASSESSED
    Reason: Not specifically addressed in this illustrative application
  Gate 3 (Documentation and Transparency): ✗ FAIL
    Reason: Proprietary constraints limiting documentation transparency
  Gate 4 (Accountability and Decision Rights): — NOT ASSESSED
    Reason: Not specifically addressed in this illustrative application
  Gate 5 (Lifecycle Monitoring and Drift Management): ✗ FAIL
    Reason: Absent alert-burden thresholds; no prespecified monitoring triggers

Summary: Under a gate-based architecture, deployment-blocking failures would likely 

### What This Illustrates

Under a gate-based architecture, three deployment-blocking failures would have surfaced before deployment:
- **Gate 1 (Safety):** External validation revealed materially lower performance than vendor claims — this is a non-overridable failure
- **Gate 3 (Documentation):** Proprietary constraints limited transparency — this is overridable but would require interim documentation and a committed timeline
- **Gate 5 (Monitoring):** No alert-burden thresholds were prespecified — this is a non-overridable failure

Two of the three failing gates (1 and 5) are non-overridable, meaning no override pathway could have circumvented them.

### What This Does NOT Demonstrate

This is a **retrospective illustrative application**, not a causal counterfactual. The manuscript explicitly notes that it does not claim gate logic would have produced categorically better patient outcomes. The companion historical replay study provides more extensive retrospective analysis with structural caveats documented in that study.


In [4]:
# Output hash for manifest validation
h = hashlib.sha256()
with open(output_path, 'rb') as f:
    h.update(f.read())
print(f"SHA-256 of {output_path.name}: {h.hexdigest()}")


SHA-256 of override_summary.csv: 2c26433216e917105bdbfa9815d80300212a4130bb757e8b9cf4e740349bfaba
